<a href="https://colab.research.google.com/github/RidwanRidu/Bengali-English-CodeMixed-Sentiment-Analysis/blob/main/Sentiment_Analysis_of_Bengali_Social_Media_Posts_using_BERT_based_Transformer_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd

url = "https://huggingface.co/datasets/Tanvir106/sentnob-bangla-dataset/raw/main/Train.csv"
df = pd.read_csv(url)

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 5 rows:")
print(df.head())

Shape: (12575, 2)

Columns: ['Data', 'Label']

First 5 rows:
                                                Data  Label
0  মুগ্ধ হয়ে গেলাম মামু. আর তোমায় কি কমু. বলো তোম...      1
1  এই কুত্তার বাচ্চাদের জন্য দেশটা আজ এমন অবস্তায়...      2
2                          ভাই আপনার কথাই যাদু রয়েছে      1
3                        উওরটা আমার অনেক ভাল লেগেছে       1
4  আমার নিজের গাড়ী নিয়ে কি সাজেক যেতে পারবো না ?...      0


In [3]:
label_col = 'Data'   # আপনার কলামের নাম অনুযায়ী বদলান
print(df[label_col].value_counts())

Data
ভাইয়া অনেক সুন্দর হইছে । ভাইয়া কয়েকদিন ধরে আপনি ♥ দেন না , ফুরিয়ে গেছে নাকি                                                                                                             3
কুত্তারবাচ্ছা , তোর জন্মই ত পতিতালয়ে । তা না হলে অন্যর মাকে গালি দিতি না । তোর মা বোন বেশ্যা । বেশ্যাগিরি করে তোদের জীবন চলে । খাংকি মাগির পোলা                                         3
খুবই ভালো জিনিস নিয়ে আপনারা আগ্রহ দেখাচ্ছেন । গান বাজনা হারাম । ছেলেটাকে গায়ক না বানিয়ে পড়াশোনায় সাহায্য করুন , সে ডাক্তার হতে চায়                                                     3
আমি কাউকে শুধু ধন্যবাদ দিতে চাই না , এতটুকু বলি যে অন্যকে সাহায্য করে তাকে আল্লাহ সাহায্য করে                                                                                           3
এই অবস্থা শুধু বুয়েটে না প্রত্যেকটা পাবলিক বিশ্ববিদ্যালয় , প্রত্যেকটা সরকারি কলেজের রেগিং নামে শারীরিক ও মানসিক অত্যাচার করা হয়                                                      3
                                                                 

In [4]:
# কলাম রিনেম
df = df.rename(columns={'Data': 'text', 'Label': 'label'})

# যদি আপনার লেবেলগুলি ইতিমধ্যেই সংখ্যায় থাকে (যেমন: 0, 1, 2), তাহলে এই ম্যাপিংয়ের প্রয়োজন নেই।
# যদি আপনি সংখ্যাগুলিকে স্ট্রিং-এ রূপান্তর করতে চান, তাহলে ম্যাপটি এরকম হতে পারে:
# label_map = {0: 'Neutral', 1: 'Positive', 2: 'Negative'}
# df['label'] = df['label'].map(label_map)

# মিসিং ভ্যালু চেক
print(df.isnull().sum())
df = df.dropna(subset=['text', 'label'])

text     0
label    0
dtype: int64


In [5]:
from datasets import Dataset

dataset = Dataset.from_pandas(df[['text', 'label']])
print(dataset)

Dataset({
    features: ['text', 'label'],
    num_rows: 12575
})


In [6]:
dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = dataset['train']
val_dataset = dataset['test']

In [7]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import torch

model_name = "sagorsarker/bangla-bert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

def tokenize_fn(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128)

train_dataset = train_dataset.map(tokenize_fn, batched=True)
val_dataset = val_dataset.map(tokenize_fn, batched=True)

# Trainer সেটআপ
training_args = TrainingArguments(
    output_dir="./sentnob_banglabert",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    from sklearn.metrics import accuracy_score, f1_score
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro')
    return {'accuracy': acc, 'f1': f1}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

config.json:   0%|          | 0.00/491 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/2.24M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  660MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: sagorsarker/bangla-bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/10060 [00:00<?, ? examples/s]

Map:   0%|          | 0/2515 [00:00<?, ? examples/s]

TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'